# Notebook 55 — Agente RAG instrumentado con MLflow

Un RAG no es una sola llamada al LLM. Tiene por lo menos dos etapas con fallos distintos:

1. **retrieval**: encontrar evidencia pertinente;
2. **generation**: responder usando esa evidencia y citarla.

MLflow Tracing registra cada etapa como un *span*. Esto permite inspeccionar si una respuesta
mala fue causada por el índice o por el generador.

El agente queda parametrizado por:

- `k`: cantidad de chunks recuperados;
- `prompt_version`: `v1` o `v2`;
- `citation_policy`: cómo convertir las citas propuestas por el modelo en citas finales.

## 1. Dependencias


In [ ]:
%pip install -q -U databricks-ai-search "mlflow[databricks]>=3.1.0" databricks-sdk
%restart_python


## 2. Configuración compartida

El notebook 56 usa `%run ./55_RAG_Agente_MLflow` para importar estas funciones. Por eso las
celdas de demostración se ejecutan únicamente cuando este notebook corre de forma directa.


In [ ]:
import json
import os
import re
import time

def _ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


_ensure_text_widget("catalogo", "big_data_ii_2025", "1. Catálogo UC")
_ensure_text_widget("esquema", "spark_examples", "2. Schema UC")
_ensure_text_widget("volume", "agenteval_squadv2", "3. Volume UC")
_ensure_text_widget(
    "endpoint_ai_search",
    "agenteval_ai_search",
    "4. Endpoint AI Search",
)
_ensure_text_widget(
    "llm_endpoint",
    "databricks-meta-llama-3-1-8b-instruct",
    "5. Foundation Model endpoint",
)

CATALOG = dbutils.widgets.get("catalogo").strip()
SCHEMA = dbutils.widgets.get("esquema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
PREFERRED_VS_ENDPOINT = dbutils.widgets.get("endpoint_ai_search").strip()
REQUESTED_LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()

for nombre, valor in {"catalogo": CATALOG, "esquema": SCHEMA, "volume": VOLUME}.items():
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido en {nombre}: {valor!r}")

VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_corpus"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_index"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
assert spark.catalog.tableExists(T_CORPUS), (
    f"No existe {T_CORPUS}. Ejecuta primero los notebooks 52 y 53."
)


def _notebook_path() -> str:
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return ctx.notebookPath().get()
    except Exception:
        return ""


CURRENT_NOTEBOOK_PATH = _notebook_path()
RUNNING_STANDALONE = "55_RAG_Agente_MLflow" in CURRENT_NOTEBOOK_PATH

print(f"Notebook actual : {CURRENT_NOTEBOOK_PATH}")
print(f"Modo            : {'directo' if RUNNING_STANDALONE else 'importado con %run'}")


## 3. Experimento por usuario y espacio temporal en un Volume

El experimento se crea bajo `/Users/<usuario>/agenteval_rag`. Así cada estudiante conserva
sus runs y trazas sin colisiones.

`MLFLOW_DFS_TMP` y `SPARKML_TEMP_DFS_PATH` apuntan a rutas dentro del Volume de Unity
Catalog. 


In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

MLFLOW_DFS_TMP = f"{VOL}/mlflow_tmp/mlflow_dfs"
SPARKML_TEMP_DFS_PATH = f"{VOL}/mlflow_tmp/sparkml"
dbutils.fs.mkdirs(MLFLOW_DFS_TMP)
dbutils.fs.mkdirs(SPARKML_TEMP_DFS_PATH)

os.environ["MLFLOW_DFS_TMP"] = MLFLOW_DFS_TMP
os.environ["SPARKML_TEMP_DFS_PATH"] = SPARKML_TEMP_DFS_PATH

import mlflow

CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]
EXPERIMENT_PATH = f"/Users/{CURRENT_USER}/agenteval_rag"
EXPERIMENT = mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Usuario          : {CURRENT_USER}")
print(f"Experimento      : {EXPERIMENT_PATH}")
print(f"Experiment ID    : {EXPERIMENT.experiment_id}")
print(f"MLflow           : {mlflow.__version__}")
print(f"MLFLOW_DFS_TMP   : {MLFLOW_DFS_TMP}")
print(f"SPARKML_TEMP_DFS : {SPARKML_TEMP_DFS_PATH}")


## 4. Conectar AI Search y el Foundation Model

El endpoint del LLM es un widget porque la disponibilidad cambia por región y fecha. El
valor predeterminado prioriza un modelo pequeño para la clase. Si la llamada devuelve
`RESOURCE_DOES_NOT_EXIST`, abre **Serving → Foundation Model APIs**, copia el nombre de un
modelo de chat habilitado y actualiza el widget.

Para AI Search se aplica la misma regla que en el notebook 54: reutilizar el endpoint único
de Free Edition.


In [ ]:
from databricks.ai_search.client import AISearchClient
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
available_serving = sorted(
    {
        endpoint.name
        for endpoint in workspace.serving_endpoints.list()
        if getattr(endpoint, "name", None)
    }
)

LLM_ENDPOINT = REQUESTED_LLM_ENDPOINT
if available_serving and LLM_ENDPOINT not in available_serving:
    print(
        f"Advertencia: '{LLM_ENDPOINT}' no aparece en serving_endpoints.list(). "
        "Los endpoints pay-per-token pueden no aparecer en todas las regiones; "
        "se probará el nombre indicado por el widget."
    )

openai_client = workspace.serving_endpoints.get_open_ai_client()

ai_search = AISearchClient(disable_notice=True)
endpoint_response = ai_search.list_endpoints()
endpoint_items = (
    endpoint_response.get("endpoints", [])
    if isinstance(endpoint_response, dict)
    else list(endpoint_response or [])
)
endpoint_names = [
    e.get("name") if isinstance(e, dict) else getattr(e, "name", None)
    for e in endpoint_items
]
endpoint_names = [name for name in endpoint_names if name]

if PREFERRED_VS_ENDPOINT in endpoint_names:
    VS_ENDPOINT = PREFERRED_VS_ENDPOINT
elif endpoint_names:
    VS_ENDPOINT = endpoint_names[0]
else:
    raise RuntimeError("No hay endpoint de AI Search. Ejecuta primero el notebook 54.")

assert ai_search.index_exists(index_name=INDEX_NAME), (
    f"No existe el índice {INDEX_NAME}. Ejecuta primero el notebook 54."
)
index = ai_search.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
index_status = index.describe().get("status", {})
assert index_status.get("ready", False), (
    f"El índice todavía no está listo: {index_status}. Vuelve al notebook 54."
)

print(f"LLM endpoint      : {LLM_ENDPOINT}")
print(f"AI Search endpoint: {VS_ENDPOINT}")
print(f"Índice            : {INDEX_NAME}")


## 5. Dos versiones del prompt

La documentación está en español, pero se pide responder en el idioma de la pregunta para
comparar la salida con la respuesta SQuAD en inglés.

- **v1**: cita los chunks utilizados.
- **v2**: añade una sola hipótesis: citar el conjunto mínimo suficiente.

El formato `CITAS: ...` permite evaluar automáticamente las citas.


In [ ]:
PROMPTS = {
    "v1": """Eres un asistente de preguntas y respuestas con grounding.
Responde ÚNICAMENTE con información contenida en los CONTEXTOS proporcionados.

Reglas:
1. Si los contextos no contienen la respuesta, responde exactamente: NO_ENCONTRADO
2. Responde de forma breve y en el mismo idioma de la pregunta.
3. Termina con una línea separada en este formato exacto:
CITAS: chunk_id_1, chunk_id_2""",

    "v2": """Eres un asistente de preguntas y respuestas con grounding.
Responde ÚNICAMENTE con información contenida en los CONTEXTOS proporcionados.

Reglas:
1. Si los contextos no contienen la respuesta, responde exactamente: NO_ENCONTRADO
2. Responde de forma breve y en el mismo idioma de la pregunta.
3. Cita el conjunto MÍNIMO de chunks suficiente para respaldar la respuesta. No cites un
   chunk solo porque sea relacionado; cítalo únicamente si la respuesta quedaría sin
   respaldo al quitarlo.
4. Termina con una línea separada en este formato exacto:
CITAS: chunk_id_1""",
}

CITATION_POLICIES = ["model", "intersect", "all_retrieved", "top1"]

print("Versiones disponibles:", sorted(PROMPTS))


### Registrar las versiones del prompt en MLflow

Cuando el notebook se ejecuta directamente se crea un run cuyo artefacto contiene ambos
prompts completos. Los runs de evaluación también registrarán `prompt_version` como
parámetro, enlazando la hipótesis con las métricas.


In [ ]:
PROMPT_RUN_ID = None
if RUNNING_STANDALONE:
    with mlflow.start_run(run_name="registro_prompts_v1_v2") as prompt_run:
        mlflow.set_tags({
            "componente": "prompt",
            "proposito": "versionamiento_prompts",
            "dataset": "squadv2_sample100",
        })
        mlflow.log_params({
            "prompt_versions": ",".join(sorted(PROMPTS)),
            "idioma_instrucciones": "es",
            "formato_citas": "CITAS: chunk_id_1, chunk_id_2",
        })
        for version, prompt_text in PROMPTS.items():
            mlflow.log_text(prompt_text, f"prompts/{version}.txt")
        mlflow.log_dict(PROMPTS, "prompts/prompts.json")
        PROMPT_RUN_ID = prompt_run.info.run_id
    print(f"Prompts registrados en el run: {PROMPT_RUN_ID}")
else:
    print("Importado desde otro notebook: se omite el run de registro de prompts.")


## 6. Retriever con span `RETRIEVER`

Este contrato es crítico. Los judges `RetrievalGroundedness` y `RetrievalRelevance` buscan
en la traza un span `RETRIEVER` cuya salida sea una lista de `Document` con `page_content`.

La función retorna **la misma lista de `Document` que registra con `span.set_outputs()`**.
Esto evita una incompatibilidad importante: algunas versiones de la instrumentación pueden
capturar automáticamente el valor retornado al cerrar el decorador. Si la función devolviera
`list[dict]` con una clave `chunk_text`, esa captura podría reemplazar el esquema manual y los
judges interpretarían cada documento como vacío porque esperan `page_content`.

Referenica: https://mlflow.org/docs/latest/genai/concepts/span/#retriever-spans


In [ ]:
from mlflow.entities import Document, SpanType

SEARCH_COLUMNS = ["chunk_id", "title", "chunk_text"]
RAG_AGENT_CONTRACT_VERSION = "retriever_documents_v2"


def _parse_search_results(result: dict) -> list[dict]:
    manifest = result.get("manifest", {}).get("columns", [])
    names = [c.get("name") for c in manifest if isinstance(c, dict)]
    rows = result.get("result", {}).get("data_array", [])
    if not names and rows:
        names = SEARCH_COLUMNS + ["score"]

    chunks = []
    for row in rows:
        item = dict(zip(names, row))
        if item.get("score") is not None:
            try:
                item["score"] = float(item["score"])
            except (TypeError, ValueError):
                pass
        chunks.append(item)
    return chunks


def _chunk_to_document(chunk: dict) -> Document:
    "Convierte una fila de AI Search al esquema canónico de MLflow."
    chunk_id = str(chunk.get("chunk_id") or "").strip()
    page_content = chunk.get("chunk_text")

    if not chunk_id:
        raise ValueError(f"AI Search devolvió un chunk sin chunk_id: {chunk}")
    if not isinstance(page_content, str) or not page_content.strip():
        raise ValueError(
            f"AI Search devolvió page_content vacío para chunk_id={chunk_id}."
        )

    metadata = {
        "doc_uri": f"uc://{T_CORPUS}/{chunk_id}",
        "chunk_id": chunk_id,
        "index_name": INDEX_NAME,
    }
    if chunk.get("title") is not None:
        metadata["title"] = str(chunk["title"])
    if chunk.get("score") is not None:
        metadata["relevance_score"] = float(chunk["score"])

    return Document(
        id=chunk_id,
        page_content=page_content,
        metadata=metadata,
    )


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve(question: str, k: int = 3) -> list[Document]:
    "Recupera k documentos y conserva el esquema RETRIEVER esperado por MLflow."
    if k < 1:
        raise ValueError("k debe ser mayor o igual a 1.")

    result = index.similarity_search(
        query_text=question,
        columns=SEARCH_COLUMNS,
        num_results=k,
    )
    chunks = _parse_search_results(result)
    documents = [_chunk_to_document(chunk) for chunk in chunks]
    if not documents:
        raise RuntimeError(
            f"AI Search no devolvió documentos para la pregunta: {question!r}"
        )

    span = mlflow.get_current_active_span()
    if span is not None:
        span.set_outputs(documents)

    # El retorno coincide con set_outputs. Aunque el decorador capture el retorno
    # automáticamente, el span seguirá teniendo page_content y metadata válidos.
    return documents


## 7. Generador con span `LLM`


In [ ]:
@mlflow.trace(span_type=SpanType.LLM)
def generate(
    question: str,
    documents: list[Document],
    prompt_version: str = "v1",
) -> tuple[str, list[str]]:
    "Genera la respuesta y extrae las citas propuestas por el modelo."
    if prompt_version not in PROMPTS:
        raise ValueError(
            f"prompt_version={prompt_version!r} no existe. Opciones: {sorted(PROMPTS)}"
        )
    if not documents:
        return "NO_ENCONTRADO", []

    context_blocks = [
        f"[{document.id}]\n{document.page_content}"
        for document in documents
    ]
    user_message = (
        "CONTEXTOS:\n\n"
        + "\n\n".join(context_blocks)
        + f"\n\nPREGUNTA:\n{question}"
    )

    raw = None
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=LLM_ENDPOINT,
                messages=[
                    {"role": "system", "content": PROMPTS[prompt_version]},
                    {"role": "user", "content": user_message},
                ],
                temperature=0.0,
                max_tokens=180,
            )
            raw = response.choices[0].message.content.strip()
            break
        except Exception:
            if attempt == 2:
                raise
            wait_seconds = 2 ** attempt
            print(f"Reintentando llamada al LLM en {wait_seconds}s...")
            time.sleep(wait_seconds)

    citation_match = re.search(
        r"(?:CITAS|CITED):\s*(.*?)\s*$",
        raw,
        flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
    )
    if citation_match:
        proposed = [
            item.strip()
            for item in citation_match.group(1).replace("\n", ",").split(",")
            if item.strip()
        ]
        answer = raw[:citation_match.start()].strip()
    else:
        proposed = []
        answer = raw

    return answer, proposed


## 8. Políticas de citación

| Política | Comportamiento |
|---|---|
| `model` | conserva las citas propuestas por el LLM |
| `intersect` | elimina IDs que no estuvieron en retrieval |
| `all_retrieved` | control ingenuo: cita todos los chunks recuperados |
| `top1` | cita solo el resultado mejor rankeado |

Separar la propuesta del modelo de la política permite distinguir problemas de prompt de
problemas de posprocesamiento.


In [ ]:
def apply_citation_policy(
    proposed: list[str],
    documents: list[Document],
    policy: str,
) -> list[str]:
    "Aplica una política determinística y elimina duplicados conservando orden."
    if policy not in CITATION_POLICIES:
        raise ValueError(f"Política inválida: {policy}. Opciones: {CITATION_POLICIES}")

    retrieved_ids = [str(document.id) for document in documents]
    retrieved_set = set(retrieved_ids)

    if policy == "model":
        selected = proposed
    elif policy == "intersect":
        selected = [chunk_id for chunk_id in proposed if chunk_id in retrieved_set]
    elif policy == "all_retrieved":
        selected = retrieved_ids
    else:  # top1
        selected = retrieved_ids[:1]

    return list(dict.fromkeys(selected))


## 9. Agente completo con traza raíz


In [ ]:
@mlflow.trace(name="rag_agent")
def rag_agent(
    question: str,
    k: int = 3,
    prompt_version: str = "v1",
    citation_policy: str = "intersect",
) -> dict:
    "Compone retrieve → generate → política de citación."
    documents = retrieve(question=question, k=k)
    answer, proposed = generate(
        question=question,
        documents=documents,
        prompt_version=prompt_version,
    )
    citations = apply_citation_policy(
        proposed=proposed,
        documents=documents,
        policy=citation_policy,
    )

    return {
        "answer": answer,
        "response": answer,
        "cited_chunk_ids": citations,
        "retrieved_chunk_ids": [str(document.id) for document in documents],
        "prompt_version": prompt_version,
        "citation_policy": citation_policy,
    }


## 10. Ejecutar un caso y revisar lo guardado

Esta sección solo corre al abrir directamente el notebook 55. Se registra un run con sus
parámetros, una traza raíz y dos spans hijos (`RETRIEVER` y `LLM`).


In [ ]:
DEMO_RUN_ID = None
if RUNNING_STANDALONE:
    demo_row = (
        spark.table(T_CORPUS)
        .select("question_id", "question", "expected_response", "chunk_id")
        .orderBy("question_id")
        .first()
    )
    demo_gold = [demo_row["chunk_id"]]

    with mlflow.start_run(run_name="demo_un_caso_rag_v2") as demo_run:
        mlflow.log_params({
            "retriever": "ai_search",
            "k": 3,
            "prompt_version": "v2",
            "citation_policy": "intersect",
            "llm_endpoint": LLM_ENDPOINT,
            "question_id": demo_row["question_id"],
        })
        demo_output = rag_agent(
            question=demo_row["question"],
            k=3,
            prompt_version="v2",
            citation_policy="intersect",
        )
        retrieved_hit = float(
            bool(set(demo_output["retrieved_chunk_ids"]) & set(demo_gold))
        )
        citation_hit = float(
            bool(set(demo_output["cited_chunk_ids"]) & set(demo_gold))
        )
        mlflow.log_metrics({
            "demo_retrieval_hit": retrieved_hit,
            "demo_citation_hit": citation_hit,
        })
        mlflow.log_dict(
            {
                "question": demo_row["question"],
                "expected_response": demo_row["expected_response"],
                "gold_chunk_ids": demo_gold,
                "output": demo_output,
            },
            "demo/result.json",
        )
        DEMO_RUN_ID = demo_run.info.run_id

    print("=" * 72)
    print(f"Pregunta  : {demo_row['question']}")
    print(f"Esperada  : {demo_row['expected_response']}")
    print(f"Generada  : {demo_output['answer']}")
    print(f"Gold      : {demo_gold}")
    print(f"Recuperado: {demo_output['retrieved_chunk_ids']}")
    print(f"Citado    : {demo_output['cited_chunk_ids']}")
    print(f"Run ID    : {DEMO_RUN_ID}")
    print("=" * 72)
else:
    print("Importado desde el notebook 56: se omite la llamada de demostración.")


### Inspección programática

La lista de documentos del span `RETRIEVER` debe incluir `page_content`, `chunk_id` y
`doc_uri`. Si falta, detente y corrige el contrato antes de ejecutar judges.


In [ ]:
if RUNNING_STANDALONE and DEMO_RUN_ID:
    trace_id = mlflow.get_last_active_trace_id()
    print(f"Última trace ID: {trace_id}")

    if trace_id:
        trace = mlflow.get_trace(trace_id)
        print(f"{'SPAN':<24} {'TIPO':<16} {'DURACIÓN ms':>12}")
        print("-" * 56)
        for span in trace.data.spans:
            duration_ms = (span.end_time_ns - span.start_time_ns) / 1e6
            print(f"{span.name:<24} {str(span.span_type):<16} {duration_ms:>12.0f}")

        retriever_spans = trace.search_spans(span_type=SpanType.RETRIEVER)
        assert retriever_spans, "La traza no contiene un span RETRIEVER."
        persisted_documents = retriever_spans[-1].outputs
        assert isinstance(persisted_documents, (list, tuple)) and persisted_documents, (
            "El span RETRIEVER no contiene una lista de documentos."
        )
        for position, document in enumerate(persisted_documents, start=1):
            page_content = (
                document.get("page_content")
                if isinstance(document, dict)
                else getattr(document, "page_content", None)
            )
            assert isinstance(page_content, str) and page_content.strip(), (
                f"Documento {position} sin page_content en el span RETRIEVER."
            )
        print(f"\nDocumentos en RETRIEVER: {len(persisted_documents)}")
        print(f"Caracteres último documento: {len(page_content)}")
        print("✓ El contexto quedó disponible para los judges de MLflow.")

    recent_runs = mlflow.search_runs(
        experiment_ids=[EXPERIMENT.experiment_id],
        max_results=10,
        order_by=["start_time DESC"],
    )
    display(recent_runs)

    try:
        display(mlflow.search_traces(run_id=DEMO_RUN_ID))
    except Exception as e:
        print(f"No se pudo listar trazas por run_id desde el SDK: {e}")


## 11. Revisar MLflow en la UI

1. En el panel derecho del notebook abre **Experiment** o navega a **Experiments**.
2. Abre `/Users/<tu_usuario>/agenteval_rag`.
3. En **Runs**, abre `registro_prompts_v1_v2` y revisa `Artifacts/prompts`.
4. Abre `demo_un_caso_rag_v2`.
5. En **Traces**, abre `rag_agent`.
6. Expande los spans `retrieve` y `generate`.
7. Comprueba qué contexto, prompt y respuesta participaron.
